# N133 · 计算几何与凸包

**目标：** 用向量关系替代不稳定斜率比较。

**先修：** N037, N126, N013。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 点积/叉积；方向；共线；线段相交；面积；矩形/圆；凸包；整数精确判定。

**配套讲解来源：** [同名逐章意见](../../comment/133_computational_geometry.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章要解决的问题是：**几何题（点在不在边界上、线段交不交叉、一堆点的外围栅栏是什么形状、最多几个点共线）在代码里怎么判，才能既正确又不受浮点误差的坑。** “学习目标”部分 的目标一句话点题："用向量关系替代不稳定斜率比较"——凡是想用 `斜率 = dy/dx` 做比较的地方，都改成整数叉积或约分方向对，精度永远精确。

给你具体到数字的例子（判断两条线段是否相交）：

- 输入：线段 A=(0,0)→(2,2)（正方形对角线）和线段 B=(0,2)→(2,0)（另一条对角线）
- 输出：`segments_intersect` 返回 True
- 为什么？用叉积看方向：从 A 的方向看过去，B 的两个端点一个在 A 的左侧、一个在右侧（叉积一负一正）；反过来从 B 看 A 也是如此——双方都"跨越"对方，必然相交。这个过程没有算过任何除法，全部整数运算。

第二个例子（凸包）：输入 6 个点 (0,0),(1,0),(2,0),(2,2),(0,2),(1,1)（正方形四角 + 底边中点 + 中心），`convex_hull` 输出 `[(0,0),(2,0),(2,2),(0,2)]`——就是能捆住所有点的最小"橡皮筋"多边形，底边中点和中心被排除在外。

## 2. 基础知识：先读懂这些词

- **叉积（cross product，二维版）**：三个点 o、a、b 的叉积 = (a−o)×(b−o)，本章 `cross(o,a,b)` 展开成 `(a.x−o.x)*(b.y−o.y) − (a.y−o.y)*(b.x−o.x)`。几何意义是"oa 和 ob 张成的有向平行四边形面积"：结果为正说明 b 在 oa 的**左边**（逆时针方向），为负在右边，为零三点**共线**。它是全章的万能判据。
- **左转 / 右转 / 共线**：沿 o→a 走到 a 后再走向 b，若 cross(o,a,b)>0 是左转（逆时针），<0 是右转（顺时针），=0 是共线。凸包算法反复问的就是这个问题。
- **线段相交判定**：两条线段"规范相交"的条件是互为跨越——从每条线段看，另一条的两个端点分居两侧（叉积符号相反）。还要补一种情况：端点恰好落在对方线段上（接触也算相交，本章约定）。
- **包围盒测试（bounding box）**：判断共线时点 p 是否在线段 ab 上，先要求坐标夹在 ab 的横纵范围之间（min ≤ p ≤ max）。这避免了对共线但相距万里的线段误判相交。
- **矩形重叠**：两个轴对齐矩形（用左下、右上两个角表示）有**正面积**交集，条件是 x、y 两个方向都 `max(下界) < min(上界)`。注意严格小于：只贴一条边不算重叠。
- **凸包（convex hull）**：包住全部点的最小凸多边形，想象一根橡皮筋套在点集外面收缩后的形状。凸包上的"角点"叫极点；落在边中间的共线点默认不保留。
- **Andrew 单调链算法**：凸包的经典做法——点按坐标排序后，分别扫出"下凸壳"和"上凸壳"两条链（每一步都保持左转，不左转就弹栈），两条链首尾相接就是完整凸包。
- **约分方向对（规范化方向）**：把 (dx,dy) 除以 gcd(dx,dy) 再统一符号约定，得到这条直线的"最简方向"。方向 (2,4)、(−1,−2)、(3,6) 都规范化成 (1,2)——代表同一条（组平行）直线，可以直接用字典数数。竖直线是 (0,1)，完全不需要"斜率无穷大"这种特例。
- **整数精确判定**：整数坐标下叉积、gcd 全是整数运算，结果精确到比特；用浮点斜率则两个本应相等的斜率可能差 10⁻¹⁶ 而判错。这是全章拒绝浮点的根本理由。

## 3. 思路推导：从小例子开始

- **Step 1：把几何问题翻译成"叉积符号"。** 左转？右转？共线？点在边的哪侧？全部归结为 cross 的正负零。
- **Step 2：线段相交 = 互跨越 + 接触兜底。** 先查双方是否互为跨越（普通交叉）；不交叉再查有没有端点落在对方线段上（接触情形，共线 + 包围盒）。
- **Step 3：凸包 = 排序 + 两条单调链。** 点排序后从左到右扫出下壳，从右到左扫出上壳；扫描中用栈维护"到目前为止的凸轮廓"，新点进来时把造成非左转的栈顶弹掉。
- **Step 4：共线计数 = 固定锚点 + 方向归一。** 枚举每个点当锚点，把其他点相对它的方向 (dx,dy) 约分成规范形式塞进计数器，同一方向的点全在一条直线上。
- **Step 5：全程整数。** 没有任何除法进入比较环节（除法只出现在 gcd 约分里，且必然整除）。

**手算演示一：互跨越判相交。** A=(0,0)→(2,2)，B=(0,2)→(2,0)。四个叉积：cross(A端点, B 的 c=(0,2))、cross(A, d=(2,0))、cross(B, a=(0,0))、cross(B, b=(2,2))。以 cross((0,0),(2,2),(0,2)) 为例：(2−0)×(2−0) − (2−0)×(0−0) = 4 > 0，而 cross((0,0),(2,2),(2,0)) = 2×0 − 2×2 = −4 < 0，一正一负——B 的两端分居 A 两侧；对称地 A 的两端也分居 B 两侧，判 True。反例 A=(0,0)→(1,0)、B=(2,0)→(3,0)（共线但不碰）：四个叉积全是 0（共线），互跨越不成立，再查包围盒——(2,0) 的 x=2 不在 [0,1] 里，四项 `_on_segment` 全 False，判 False。

**手算演示二：凸包的单调链扫一遍**（“运行示例”部分 的输入 points=[(0,0),(1,0),(2,0),(2,2),(0,2),(1,1)]）。排序去重后按字典序：(0,0),(0,2),(1,0),(1,1),(2,0),(2,2)。扫下壳：

| 动作 | 栈内容 | 说明 |
|------|--------|------|
| 进 (0,0),(0,2) | (0,0),(0,2) | 先垫两个点 |
| 看 (1,0)：cross((0,0),(0,2),(1,0))=−2≤0 | (0,0),(1,0) | 右转，弹掉 (0,2) |
| 进 (1,1)：cross=+1 左转 | (0,0),(1,0),(1,1) | 保留 |
| 看 (2,0)：cross((1,0),(1,1),(2,0))=−1 弹 (1,1)；cross((0,0),(1,0),(2,0))=0 也弹 (1,0) | (0,0),(2,0) | 连弹两个非左转 |
| 进 (2,2)：左转 | (0,0),(2,0),(2,2) | 下壳完成，去尾得 (0,0),(2,0) |

对称地从右到左扫出上壳 (2,2),(0,2)。拼起来 hull = [(0,0),(2,0),(2,2),(0,2)]——四角点，逆时针排列。

**手算演示三：“运行示例”部分 的两张表格数据。** 三个版本的成员资格对照：

| 输入点 | 极点凸包成员 | 包含共线边界时成员 |
|--------|-------------|---------------------|
| (0,0) | True | True |
| (1,0) | False | True |
| (2,0) | True | True |
| (2,2) | True | True |
| (0,2) | True | True |
| (1,1) | False | False |

默认模式只收四个角（底边中点 (1,0) 被丢掉）；`include_collinear=True` 时 (1,0) 回到边界上，但中心点 (1,1) 无论如何都不在边界。第二张表把 (1,1) 对凸包四条有向边逐条算叉积：cross((0,0),(2,0),(1,1))=2、cross((2,0),(2,2),(1,1))=2、cross((2,2),(0,2),(1,1))=2、cross((0,2),(0,0),(1,1))=2——全部为正，即内部点在每条有向边的左侧，这正是"逆时针凸包 + 内点恒在左侧"的现场验证。

**手算演示四：max_points([(1,1),(1,1),(2,2)])。** 锚点取第一个 (1,1)：遇到第二个 (1,1) 时 dx=dy=0，记入 duplicates=1；遇到 (2,2) 时 dx=dy=1，gcd=1，方向 (1,1) 计 1。best = 1 + 1 + 1 = 3——两个重复点加上 (2,2) 全在一条直线上，答案 3。

## 4. 核心代码：cross

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def cross(o, a, b):
    return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])
```

### 逐段读懂代码

“分段实现”部分 有五个函数（含一个下划线开头的内部助手）。

**函数一：`cross(o, a, b)` —— 万物之基：叉积。**

```python
def cross(o, a, b):
    return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])
```

- 一行公式：先算两个向量 oa=(a−o)、ob=(b−o)，再算 x₁y₂ − y₁x₂。结果带符号：正 = b 在 oa 左侧，负 = 右侧，零 = 共线。整数进整数出，永不丢精度。

**函数二：`_on_segment(a, b, p)` —— 共线时点是否落在线段上。**

```python
def _on_segment(a, b, p):
    return cross(a, b, p) == 0 and min(a[0], b[0]) <= p[0] <= max(a[0], b[0]) and (min(a[1], b[1]) <= p[1] <= max(a[1], b[1]))
```

- 三个条件用 and 串联：先叉积为零（p 与 ab 共线），再检查 p 的横坐标和纵坐标都夹在 ab 的 min/max 之间（包围盒）。共线但飘在线段延长线上的点会被第二个条件拦住。`min(...)<=p[0]<=max(...)` 是 Python 的链式比较，一次表达"夹在中间"。

**函数三：`segments_intersect(a,b,c,d)` —— 线段 ab 与 cd 是否相交。**

```python
def segments_intersect(a, b, c, d):
    x, y, z, w = (cross(a, b, c), cross(a, b, d), cross(c, d, a), cross(c, d, b))
    if (x > 0 and y < 0 or (x < 0 and y > 0)) and (z > 0 and w < 0 or (z < 0 and w > 0)):
        return True
    return _on_segment(a, b, c) or _on_segment(a, b, d) or _on_segment(c, d, a) or _on_segment(c, d, b)
```

- 第一行一次算四个叉积：x、y 是 cd 两端相对 ab 的方向；z、w 是 ab 两端相对 cd 的方向。
- if 那行是"互跨越"条件：`(x>0 and y<0 or x<0 and y>0)` 说"c、d 分居 ab 两侧"（异号），后面括号对称地说"a、b 分居 cd 两侧"；两组同时成立才返回 True。注意用的都是严格大于/小于——恰好共线的情形交给下一行。
- return 那行兜底四种"接触"：c 或 d 恰好落在 ab 上（共线且在包围盒内），或 a、b 落在 cd 上。本章约定接触也算相交，"十字交叉 + 端点搭接 + 共线重叠"三种都覆盖了。

**函数四：`rectangles_overlap(a, b)` —— 轴对齐矩形是否正面积重叠。**

```python
def rectangles_overlap(a, b):
    return max(a[0], b[0]) < min(a[2], b[2]) and max(a[1], b[1]) < min(a[3], b[3])
```

- 每个矩形用 [x左, y下, x右, y上] 表示。x 方向重叠的条件是"两个左边的较大者"仍在"两个右边的较小者"左边（严格小于），y 方向同理；两个方向同时满足才有正面积的交集。严格小于意味着只贴边（比如 [0,0,1,1] 和 [1,0,2,1] 共享一条竖边）返回 False——这是本题的契约，“正确性与复杂度”部分 明确写了"只有正面积交叠才返回 True"。

**函数五：`convex_hull(points, include_collinear=False)` —— 单调链凸包。**

```python
def convex_hull(points, include_collinear=False):
    points = sorted(set(map(tuple, points)))
    if len(points) <= 2:
        return points
    if all((cross(points[0], points[-1], p) == 0 for p in points)):
        return points if include_collinear else [points[0], points[-1]]

    def half(sequence):
        stack = []
        for p in sequence:
            while len(stack) >= 2 and (cross(stack[-2], stack[-1], p) < 0 if include_collinear else cross(stack[-2], stack[-1], p) <= 0):
                stack.pop()
            stack.append(p)
        return stack
    return half(points)[:-1] + half(reversed(points))[:-1]
```

- `points=sorted(set(map(tuple,points)))`：先 `map(tuple)` 把列表点转成元组（可哈希、可排序），`set` 去重，`sorted` 字典序排序。去重是后面逻辑的前提（重复点会搞乱栈判断）。
- `if len(points)<=2: return points`：一两个点自成凸包，直接返回。
- `if all(cross(points[0],points[-1],p)==0 ...)`：全员共线的退化情形——凸包退化成一条线段，默认只返回两个端点（极点），`include_collinear=True` 时原样返回所有点。
- 内嵌 `half(sequence)`：单条链的扫描。栈里存当前链；新点 p 来了就看栈顶两个点与 p 构成什么转弯：默认模式 `cross(...)<=0`（非严格左转，即右转**或共线**）就弹栈，极点模式下只弹严格右转 `cross(...)<0`（保留共线点）。这个三元条件表达式 `<0 if include_collinear else <=0` 一行切换两种模式——默认把共线的中间点弹掉，只留角。
- `return half(points)[:-1]+half(reversed(points))[:-1]`：正序扫出下壳、逆序扫出上壳，各去掉最后一个点（两条链的端点重复，拼接时各让一个），首尾相接得逆时针凸包。

**函数六：`max_points(points)` —— 最多共线点数。**

```python
def max_points(points):
    best = 0
    for i, (x, y) in enumerate(points):
        slopes = Counter()
        duplicates = 0
        for u, v in points[i + 1:]:
            dx, dy = (u - x, v - y)
            if dx == dy == 0:
                duplicates += 1
                continue
            divisor = gcd(dx, dy)
            dx //= divisor
            dy //= divisor
            if dx < 0 or (dx == 0 and dy < 0):
                dx, dy = (-dx, -dy)
            slopes[dx, dy] += 1
        best = max(best, 1 + duplicates + max(slopes.values(), default=0))
    return best
```

- `for i,(x,y) in enumerate(points)`：外层枚举"锚点"，只看它后面的点（`points[i+1:]`），每条直线只需要被锚在编号最小的那个点上数一次。
- `dx,dy=u-x,v-y`：相对锚点的方向向量。
- `if dx==dy==0: duplicates+=1; continue`：和锚点重合的点，单独记数（它们在哪条直线上都行）。
- `divisor=gcd(dx,dy); dx//=divisor; dy//=divisor`：约分。同一方向不管长短（(2,4) 和 (3,6)）都归一到最简整数比 (1,2)。
- `if dx<0 or dx==0 and dy<0: dx,dy=-dx,-dy`：符号归一。同一条直线从锚点看可能往左也可能往右（方向 (1,2) 和 (−1,−2)），统一翻成"向右半边"（dx>0，或 dx=0 且 dy>0 竖直向上），一行消除二义性。`or`/`and` 的优先级是 and 更紧，所以读作 `dx<0 或 (dx==0 且 dy<0)`。
- `slopes[(dx,dy)]+=1`：规范方向当字典键计数，方向相同 = 共线。
- `best=max(best,1+duplicates+max(slopes.values(),default=0))`：过锚点的最优直线 = 锚点自己（那个 1）+ 重复点 + 同一方向最多的点。`default=0` 伺候锚点之后没有别的点时 `slopes` 为空的情形，不让 `max()` 报错。

## 5. 分段实现：按顺序运行

**本章接口：** `cross(o, a, b)`、`segments_intersect(a,b,c,d)`、`convex_hull(points)`、`max_points(points)`

### 导入本章使用的库

In [1]:
from math import gcd
from collections import Counter

### cross

In [2]:
def cross(o, a, b):
    return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

### _on_segment

In [3]:
def _on_segment(a, b, p):
    return cross(a, b, p) == 0 and min(a[0], b[0]) <= p[0] <= max(a[0], b[0]) and (min(a[1], b[1]) <= p[1] <= max(a[1], b[1]))

### segments_intersect

In [4]:
def segments_intersect(a, b, c, d):
    x, y, z, w = (cross(a, b, c), cross(a, b, d), cross(c, d, a), cross(c, d, b))
    if (x > 0 and y < 0 or (x < 0 and y > 0)) and (z > 0 and w < 0 or (z < 0 and w > 0)):
        return True
    return _on_segment(a, b, c) or _on_segment(a, b, d) or _on_segment(c, d, a) or _on_segment(c, d, b)

### rectangles_overlap

In [5]:
def rectangles_overlap(a, b):
    return max(a[0], b[0]) < min(a[2], b[2]) and max(a[1], b[1]) < min(a[3], b[3])

### convex_hull

In [6]:
def convex_hull(points, include_collinear=False):
    points = sorted(set(map(tuple, points)))
    if len(points) <= 2:
        return points
    if all((cross(points[0], points[-1], p) == 0 for p in points)):
        return points if include_collinear else [points[0], points[-1]]

    def half(sequence):
        stack = []
        for p in sequence:
            while len(stack) >= 2 and (cross(stack[-2], stack[-1], p) < 0 if include_collinear else cross(stack[-2], stack[-1], p) <= 0):
                stack.pop()
            stack.append(p)
        return stack
    return half(points)[:-1] + half(reversed(points))[:-1]

### max_points

In [7]:
def max_points(points):
    best = 0
    for i, (x, y) in enumerate(points):
        slopes = Counter()
        duplicates = 0
        for u, v in points[i + 1:]:
            dx, dy = (u - x, v - y)
            if dx == dy == 0:
                duplicates += 1
                continue
            divisor = gcd(dx, dy)
            dx //= divisor
            dy //= divisor
            if dx < 0 or (dx == 0 and dy < 0):
                dx, dy = (-dx, -dy)
            slopes[dx, dy] += 1
        best = max(best, 1 + duplicates + max(slopes.values(), default=0))
    return best

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [8]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

points=[(0,0),(1,0),(2,0),(2,2),(0,2),(1,1)]; hull=convex_hull(points)
show_table(['输入点','极点凸包成员','包含共线边界时成员'],[(p,p in hull,p in convex_hull(points,True)) for p in points])
show_table(['凸包有向边','内部测试点(1,1)叉积'],[((a,b),cross(a,b,(1,1))) for a,b in zip(hull,hull[1:]+hull[:1])])

输入点,极点凸包成员,包含共线边界时成员
"(0, 0)",True,True
"(1, 0)",False,True
"(2, 0)",True,True
"(2, 2)",True,True
"(0, 2)",True,True
"(1, 1)",False,False


凸包有向边,"内部测试点(1,1)叉积"
"((0, 0), (2, 0))",2
"((2, 0), (2, 2))",2
"((2, 2), (0, 2))",2
"((0, 2), (0, 0))",2


## 7. 正确性、适用条件与复杂度

叉积等于有向平行四边形面积。凸包栈删除非左转点时，该点位于相邻两点与当前点包围的内侧，不能成为该半边的极点；上下两条单调链合成逆时针凸包。方向对用GCD约分并统一符号，竖直线和重复点均不需要浮点特例斜率。

**代价：** 凸包O(n log n)时间O(n)空间；最多共线点O(n²)次方向处理、O(n)辅助。默认凸包只保留极点，include_collinear=True返回所有边界输入点（适配围栏类题）。线段接触算相交；矩形只有正面积交叠才返回True。

### 展开理解

**叉积符号为什么可靠？** 叉积的绝对值就是 oa、ob 两条边张成的平行四边形面积，符号表示张开的转向：b 在 oa 左侧时逆时针张、面积为正；右侧顺时针、取负；共线时面积为零。面积是整数坐标下乘加减出来的整数，判断精确无误。

**互跨越判定为什么充分且必要？** 两条线段规范相交时，每条线段的两端必然分居另一条线段延长线的两侧（符号异号），这就是 if 那行；反过来符号异号保证了真正的穿过。唯一漏掉的情形是"端点恰好落在线段所在直线上"（某个叉积为 0），这时互跨越的严格异号不成立，但接触确实发生了，所以用四个 `_on_segment` 兜住——"严格异号或端点搭在线段上"覆盖了平面上线段相交的所有可能位置关系。

**单调链为什么得到凸包？** 扫描时栈维护的不变性质是"栈里任意连续三点都严格左转"（不满足就弹掉破坏者）。被弹掉的点为什么永远不亏？它处在栈中相邻两点与新点构成的"内侧"，被新边界包在里面，不可能是这个半边的极点。下壳和上壳各自从最左扫到最右、再从最右扫回最左，两条链在两个极端点会合，合成完整的逆时针凸包。“正确性与复杂度”部分 还说明：“运行示例”部分 第二张表"内点对所有边叉积为正"正是这个逆时针方向的体现。

**方向对为什么无浮点特例？** 斜率 dy/dx 在 dx=0（竖线）时无定义、在接近竖线时浮点误差爆炸；约分后的方向对 (dx,dy) 对竖线就是 (0,1)，一切平线、斜线、竖线统一处理，重复点也走 duplicates 分支，从根上避开分类讨论。

**复杂度（结合具体数字）：** 凸包排序 O(n log n) + 两次线性扫描（每个点最多进栈一次出栈一次）O(n)，总计 O(n log n)——n = 10⁵ 时排序约 10⁵×17 ≈ 2×10⁶ 次比较，一眨眼。`max_points` 双重循环 O(n²) 次方向处理——n = 1000 时约 50 万对，毫秒级；n = 10⁴ 时约 5×10⁷ 对，秒级可忍，再大就要换极角排序思路。空间都是 O(n) 辅助。

**两个契约再念一遍：** 凸包默认只留极点，`include_collinear=True` 留下所有边界点（“正确性与复杂度”部分 提到这适配"围栏要穿过所有边界桩"类题目）；矩形必须正面积重叠才算，贴边不算。

## 8. 单元测试：每个用例在检查什么

1. `assert segments_intersect((0,0),(2,2),(0,2),(2,0))` —— **正常值**：正方形两条对角线规范交叉，走互跨越分支。
2. `assert segments_intersect((0,0),(1,0),(1,0),(2,0))` —— **接触边界**：两条共线线段在端点 (1,0) 相接，互跨越分支四个叉积全 0 不成立，靠 `_on_segment` 兜底返回 True。
3. `assert not segments_intersect((0,0),(1,0),(2,0),(3,0))` —— **共线不相交**：同一直线上分离的两段，共线但包围盒夹不住对方端点，四项兜底全 False。
4. `assert not rectangles_overlap([0,0,1,1],[1,0,2,1])` —— **贴边边界**：两矩形共享一条竖边，x 方向 `max(0,1)=1 < min(1,2)=1` 不成立（1 不小于 1），按契约返回 False。
5. `assert max_points([(1,1),(1,1),(2,2)])==3` —— **重复点特例**：两个重合点加一个共线点，检验 duplicates 计数和 `1+duplicates+max(...)` 的组装。
6. `assert convex_hull([(0,0),(1,1),(2,2)])==[(0,0),(2,2)]` —— **全共线退化**：凸包退成线段，默认只回两个端点，中间的 (1,1) 被丢。
7. `assert len(convex_hull([(0,0),(1,1),(2,2)],True))==3` —— 同一输入开 `include_collinear`，三个点全保留，验证开关真的切换了弹栈条件。
8. 250 轮随机对账（seed=133）：每轮在 [−3,3]² 里造 1~9 个随机点。凸包验证三件事——若 hull ≥ 3 个点，则每个输入点对每条有向边的叉积 ≥ 0（全点在左侧或边上，`cross(a,b,p)>=0`），且凸包顶点自身连续三点严格左转（`cross(hull[i-2],hull[i-1],hull[i])>0`，即没有平角顶点混进默认模式）；若 hull 恰好 2 个点，全部输入点必须落在两点线段上；若只有 1 个点，所有输入必为同一点。`max_points` 则和双层循环暴力对账（重数最多的点数 vs 每对点连线上数点），250 轮全绿。

In [9]:
assert segments_intersect((0, 0), (2, 2), (0, 2), (2, 0))

assert segments_intersect((0, 0), (1, 0), (1, 0), (2, 0))

assert not segments_intersect((0, 0), (1, 0), (2, 0), (3, 0))

assert not rectangles_overlap([0, 0, 1, 1], [1, 0, 2, 1])

assert max_points([(1, 1), (1, 1), (2, 2)]) == 3

assert convex_hull([(0, 0), (1, 1), (2, 2)]) == [(0, 0), (2, 2)]

assert len(convex_hull([(0, 0), (1, 1), (2, 2)], True)) == 3

from random import Random

rng = Random(133)

for _ in range(250):
    points = [(rng.randrange(-3, 4), rng.randrange(-3, 4)) for _ in range(rng.randrange(1, 10))]
    hull = convex_hull(points)
    if len(hull) >= 3:
        assert all((cross(a, b, p) >= 0 for a, b in zip(hull, hull[1:] + hull[:1]) for p in points))
        assert all((cross(hull[i - 2], hull[i - 1], hull[i]) > 0 for i in range(len(hull))))
    elif len(hull) == 2:
        assert all((_on_segment(hull[0], hull[1], p) for p in points))
    else:
        assert len(set(points)) == 1
    want = max(Counter(points).values())
    for i, a in enumerate(points):
        for b in points[i + 1:]:
            if a != b:
                want = max(want, sum((cross(a, b, p) == 0 for p in points)))
    assert max_points(points) == want

print('N133: 所有本章断言通过')

N133: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 保留所有边界共线点。

**练习 2：** 用规范化gcd方向替代浮点斜率。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（保留所有边界共线点）：** 提示——关键在弹栈条件从 `<=0` 放宽到 `<0`：默认模式把"共线（=0）"和"右转（<0）"一起弹，所以中间的共线点消失；只弹严格右转，共线点就能留在栈里。但要注意随之而来的新边界：全共线输入时两种模式要分别返回什么？（“分段实现”部分 已给答案：默认返回两端点、开启返回全部。）用 (0,0),(1,0),(2,0),(2,2),(0,2) 这种"底边带中点"的例子手算一遍两种输出，再和 `convex_hull(points, True)` 对照。

**练习 2（规范化 gcd 方向替代浮点斜率）：** 提示——自己写一个"斜率字符串"版本（`f"{dy}/{dx}"` 当字典键）和本章的方向对版本对比：竖线 (dx=0) 在斜率版里怎么表示？(1,2) 和 (2,4) 在斜率版里字符串不同、会漏算，而 gcd 约分后都是 (1,2)。手算 max_points([(0,0),(1,1),(2,2),(0,1)])：锚 (0,0) 时方向 (1,1) 出现两次、(0,1) 一次，得 1+0+2=3；锚 (0,1) 时 (1,0) 方向两次又得 3——最终答案 3，两点各验一遍。注意符号归一步骤漏掉的话，(1,−1) 和 (−1,1) 会被当两条线，这是最容易写错的角落。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [10]:
# 完整实现：本单元格不依赖其他单元格。
from math import gcd

from collections import Counter

def cross(o, a, b):
    return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

def _on_segment(a, b, p):
    return cross(a, b, p) == 0 and min(a[0], b[0]) <= p[0] <= max(a[0], b[0]) and (min(a[1], b[1]) <= p[1] <= max(a[1], b[1]))

def segments_intersect(a, b, c, d):
    x, y, z, w = (cross(a, b, c), cross(a, b, d), cross(c, d, a), cross(c, d, b))
    if (x > 0 and y < 0 or (x < 0 and y > 0)) and (z > 0 and w < 0 or (z < 0 and w > 0)):
        return True
    return _on_segment(a, b, c) or _on_segment(a, b, d) or _on_segment(c, d, a) or _on_segment(c, d, b)

def rectangles_overlap(a, b):
    return max(a[0], b[0]) < min(a[2], b[2]) and max(a[1], b[1]) < min(a[3], b[3])

def convex_hull(points, include_collinear=False):
    points = sorted(set(map(tuple, points)))
    if len(points) <= 2:
        return points
    if all((cross(points[0], points[-1], p) == 0 for p in points)):
        return points if include_collinear else [points[0], points[-1]]

    def half(sequence):
        stack = []
        for p in sequence:
            while len(stack) >= 2 and (cross(stack[-2], stack[-1], p) < 0 if include_collinear else cross(stack[-2], stack[-1], p) <= 0):
                stack.pop()
            stack.append(p)
        return stack
    return half(points)[:-1] + half(reversed(points))[:-1]

def max_points(points):
    best = 0
    for i, (x, y) in enumerate(points):
        slopes = Counter()
        duplicates = 0
        for u, v in points[i + 1:]:
            dx, dy = (u - x, v - y)
            if dx == dy == 0:
                duplicates += 1
                continue
            divisor = gcd(dx, dy)
            dx //= divisor
            dy //= divisor
            if dx < 0 or (dx == 0 and dy < 0):
                dx, dy = (-dx, -dy)
            slopes[dx, dy] += 1
        best = max(best, 1 + duplicates + max(slopes.values(), default=0))
    return best

# 示例与回归测试
assert segments_intersect((0, 0), (2, 2), (0, 2), (2, 0))

assert segments_intersect((0, 0), (1, 0), (1, 0), (2, 0))

assert not segments_intersect((0, 0), (1, 0), (2, 0), (3, 0))

assert not rectangles_overlap([0, 0, 1, 1], [1, 0, 2, 1])

assert max_points([(1, 1), (1, 1), (2, 2)]) == 3

assert convex_hull([(0, 0), (1, 1), (2, 2)]) == [(0, 0), (2, 2)]

assert len(convex_hull([(0, 0), (1, 1), (2, 2)], True)) == 3

from random import Random

rng = Random(133)

for _ in range(250):
    points = [(rng.randrange(-3, 4), rng.randrange(-3, 4)) for _ in range(rng.randrange(1, 10))]
    hull = convex_hull(points)
    if len(hull) >= 3:
        assert all((cross(a, b, p) >= 0 for a, b in zip(hull, hull[1:] + hull[:1]) for p in points))
        assert all((cross(hull[i - 2], hull[i - 1], hull[i]) > 0 for i in range(len(hull))))
    elif len(hull) == 2:
        assert all((_on_segment(hull[0], hull[1], p) for p in points))
    else:
        assert len(set(points)) == 1
    want = max(Counter(points).values())
    for i, a in enumerate(points):
        for b in points[i + 1:]:
            if a != b:
                want = max(want, sum((cross(a, b, p) == 0 for p in points)))
    assert max_points(points) == want

print('N133: 所有本章断言通过')

N133: 所有本章断言通过


## 11. 代表题与迁移

- **149. Max Points on a Line（直线上最多的点数）**：`max_points` 的原型题（canonical），练"锚点 + gcd 约分方向对"这个无浮点共线计数套路。
- **587. Erect the Fence（安装栅栏）**：给树桩求围栏——凸包原型题（canonical），而且它要求边界上的桩也要出现在答案里，正好对应 `include_collinear=True` 模式。
- **836. Rectangle Overlap（矩形重叠）**：轴对齐矩形是否相交——`rectangles_overlap` 的原型题（canonical），练"投影区间 max/min 比较"和贴边不算的边界。
- **223. Rectangle Area（矩形面积）**：求两矩形覆盖的总面积——同一套矩形交叠判定再乘上"交集宽 × 高"的面积计算（canonical），是 836 的进阶版。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。